In [2]:
import requests
from bs4 import BeautifulSoup


In [ ]:
import requests
from bs4 import BeautifulSoup
import csv

# Base URL do site UFC Stats
BASE_URL = "http://ufcstats.com/statistics/fighters?char=a"

# Função para extrair informações de uma página
def scrape_page(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")
    
    # Encontrar todas as linhas da tabela
    rows = soup.select("table.b-statistics__table tbody tr.b-statistics__table-row")
    
    # Dados extraídos
    fighters_data = []
    for row in rows:
        # Encontrar todas as colunas (td)
        cols = row.find_all("td")
        
        # Ignorar linhas vazias ou com menos colunas do que o esperado
        if len(cols) < 11:
            continue

        # Extrair os dados e armazenar em um dicionário
        fighters_data.append({
            "First": cols[0].get_text(strip=True),
            "Last": cols[1].get_text(strip=True),
            "Nickname": cols[2].get_text(strip=True),
            "Ht.": cols[3].get_text(strip=True),
            "Wt.": cols[4].get_text(strip=True),
            "Reach": cols[5].get_text(strip=True),
            "Stance": cols[6].get_text(strip=True),
            "W": cols[7].get_text(strip=True),
            "L": cols[8].get_text(strip=True),
            "D": cols[9].get_text(strip=True),
            "Belt": cols[10].get_text(strip=True),
        })
    return fighters_data

# Função para encontrar os links de paginação
def get_pagination_links(base_url):
    response = requests.get(base_url)
    soup = BeautifulSoup(response.text, "html.parser")
    links = [a["href"] for a in soup.select(".b-statistics__paginate a")]
    return links

# Scraping completo com paginação
def scrape_all_pages(base_url):
    pagination_links = get_pagination_links(base_url)
    all_fighters = []

    for link in pagination_links:
        print(f"Scraping: {link}")
        fighters = scrape_page(link)
        all_fighters.extend(fighters)

    return all_fighters

# Escrever dados em um arquivo CSV
def save_to_csv(data, filename="ufc_fighters.csv"):
    if data:
        keys = data[0].keys()
        with open(filename, "w", newline="", encoding="utf-8") as file:
            writer = csv.DictWriter(file, fieldnames=keys)
            writer.writeheader()
            writer.writerows(data)
    else:
        print("Nenhum dado para salvar.")

# Main
if __name__ == "__main__":
    print("Iniciando o scraping...")
    all_fighters = scrape_all_pages(BASE_URL)
    print(f"Total de lutadores extraídos: {len(all_fighters)}")
    save_to_csv(all_fighters)
    print("Dados salvos em 'ufc_fighters.csv'.")


In [1]:
import pandas as pd

In [7]:
df = pd.read_csv('/home/gassuncao/mma-status/mma-stats/data/raw/ufc_athletes.csv')

In [19]:
df[df["Athlete ID"] == "mario-bautista"]["Significant Strikes Landed"]

210    542.0
Name: Significant Strikes Landed, dtype: float64

In [ ]:
df[df['First'] == 'Karolina']

In [14]:
import requests
from bs4 import BeautifulSoup
import csv

# Base URL do site UFC Stats
url = "https://www.ufc.com.br/athlete/umar-nurmagomedov"


response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

In [ ]:
print(response.text)

In [ ]:
import requests
from bs4 import BeautifulSoup
import csv
import time

# URL principal para os atletas
BASE_URL = "https://www.ufc.com.br/athletes/all"

# Função para extrair os dados de cada atleta
def scrape_athletes_from_page(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")
    
    # Encontrando todos os atletas na página
    athletes = soup.find_all("div", class_="c-listing-athlete-flipcard__inner")
    
    all_athletes = []
    
    for athlete in athletes:
        try:
            athlete_id = athlete.find("a", class_="e-button--black")
            athlete_id = athlete_id.get("href")
            athlete_id = athlete_id.split("/")[-1]
            # Nome do atleta
            name = athlete.find("span", class_="c-listing-athlete__name")
            name = name.get_text(strip=True) if name else "N/A"
            
            # Apelido do atleta
            nickname = athlete.find("span", class_="c-listing-athlete__nickname")
            nickname = nickname.get_text(strip=True) if nickname else "N/A"
            
            # Record (Vitórias, Derrotas e Empates)
            record = athlete.find("span", class_="c-listing-athlete__record")
            record = record.get_text(strip=True) if record else "N/A"
            
            # Separando vitórias, derrotas e empates
            wins, losses, draws = record.replace(" (V-D-E)", "").split("-") if record != "N/A" else ("N/A", "N/A", "N/A")
            if record != "N/A":
                try:
                    wins, losses, draws = record.split('-')
                except ValueError:
                    pass  # Caso não consiga dividir, mantém os valores como "N/A"
            
            # Categoria de peso
            weight_class = athlete.find("div", class_="field--name-stats-weight-class")
            try:
                weight_class = weight_class.find("div", class_="field__item")
                weight_class = weight_class.get_text(strip=True) if weight_class else "N/A"
            except:
                weight_class = "N/A"
            
            # Organizando os dados do atleta
            athlete_data = {
                "Athlete ID": athlete_id,
                "Name": name,
                "Nickname": nickname,
                "Wins": wins,
                "Losses": losses,
                "Draws": draws,
                "Weight Class": weight_class
            }
            
            all_athletes.append(athlete_data)
        except Exception as e:
            print(f"Erro ao processar atleta {athlete}: {e}")
            continue
    
    return all_athletes

# Função para percorrer as páginas e coletar todos os atletas
def scrape_all_athletes(base_url):
    all_athletes = []
    page = 1
    
    while True:
        print(f"Scraping page {page}...")
        url = f"{base_url}?page={page}"
        athletes = scrape_athletes_from_page(url)
        
        if not athletes:
            break
        
        all_athletes.extend(athletes)
        page += 1
        
        # Pausar entre requisições para evitar sobrecarga no servidor
        time.sleep(2)  # Atraso de 2 segundos entre requisições
        
    return all_athletes

# Função para salvar os dados em um arquivo CSV
def save_to_csv(data, filename="ufc_athletes.csv"):
    if data:
        keys = data[0].keys()
        with open(filename, "w", newline="", encoding="utf-8") as file:
            writer = csv.DictWriter(file, fieldnames=keys)
            writer.writeheader()
            writer.writerows(data)
    else:
        print("Nenhum dado para salvar.")

# Main
if __name__ == "__main__":
    print("Iniciando o scraping...")
    all_athletes = scrape_all_athletes(BASE_URL)
    print(f"Total de atletas extraídos: {len(all_athletes)}")
    save_to_csv(all_athletes)
    print("Dados salvos em 'ufc_athletes.csv'.")


In [ ]:
import pandas as pd
df = pd.read_csv('ufc_athletes.csv')

# Soma das colunas Wins e Losses
wins_losses_sum = pd.DataFrame({
    'Total': [df['Wins'].sum(), df['Losses'].sum()]
}, index=['Wins', 'Losses'])

print("\nSoma total de vitórias e derrotas:")
print(wins_losses_sum)



In [1]:
import requests
from bs4 import BeautifulSoup
import csv
import time
### Versao top
# URL principal para os atletas
BASE_URL = "https://www.ufc.com.br/athletes/all"

# Função para extrair os dados de cada atleta
def scrape_athletes_from_page(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")
    
    # Encontrando todos os atletas na página
    athletes = soup.find_all("div", class_="c-listing-athlete-flipcard__inner")
    
    all_athletes = []
    
    for athlete in athletes:
        try:
            athlete_id = athlete.find("a", class_="e-button--black")
            athlete_id = athlete_id.get("href")
            athlete_url = f"https://www.ufc.com.br{athlete_id}"
            athlete_id = athlete_id.split("/")[-1]
            
            # Nome do atleta
            name = athlete.find("span", class_="c-listing-athlete__name")
            name = name.get_text(strip=True) if name else "N/A"
            
            # Apelido do atleta
            nickname = athlete.find("span", class_="c-listing-athlete__nickname")
            nickname = nickname.get_text(strip=True) if nickname else "N/A"
            
            # Record (Vitórias, Derrotas e Empates)
            record = athlete.find("span", class_="c-listing-athlete__record")
            record = record.get_text(strip=True) if record else "N/A"
            
            # Separando vitórias, derrotas e empates
            wins, losses, draws = record.replace(" (V-D-E)", "").split("-") if record != "N/A" else ("N/A", "N/A", "N/A")
            
            # Categoria de peso
            weight_class = athlete.find("div", class_="field--name-stats-weight-class")
            try:
                weight_class = weight_class.find("div", class_="field__item")
                weight_class = weight_class.get_text(strip=True) if weight_class else "N/A"
            except:
                weight_class = "N/A"
            
            # Dados adicionais do perfil
            athlete_profile = scrape_athlete_profile(athlete_url)
            
            # Organizando os dados do atleta
            athlete_data = {
                "Athlete ID": athlete_id,
                "Name": name,
                "Nickname": nickname,
                "Wins": wins,
                "Losses": losses,
                "Draws": draws,
                "Weight Class": weight_class,
                **athlete_profile
            }
            
            all_athletes.append(athlete_data)
        except Exception as e:
            print(f"Erro ao processar atleta {athlete}: {e}")
            continue
    
    return all_athletes

def scrape_athlete_profile(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")
    
    try:
        # Informações detalhadas
        stats = {}
        
        overlap_stats = soup.find_all("dl", class_="c-overlap__stats")
        for stat in overlap_stats:
            label = stat.find("dt").text.strip()
            value = stat.find("dd").text.strip()
            
            if label == "Golpes Sig. Conectados":
                stats["Significant Strikes Landed"] = value
            elif label == "Golpes Sig. Desferidos":
                stats["Significant Strikes Attempted"] = value
            elif label == "Quedas aplicadas":
                stats["Takedowns Landed"] = value
            elif label == "Tentativas de queda":
                stats["Takedowns Attempted"] = value

        bio = soup.find_all("div", class_="c-bio__row--3col")
        if bio:
            # Primeira linha de informações
            try:
                stats["Age"] = bio[0].find("div", class_="field__item").text.strip() if bio[0].find("div", class_="field__item") else "N/A"
            except:
                stats["Age"] = "N/A"
            try:
                stats["Height"] = bio[0].find_all("div", class_="c-bio__text")[1].text.strip() if bio[0].find_all("div", class_="c-bio__text")[1] else "N/A"
            except:
                stats["Height"] = "N/A"
            try:
                stats["Weight"] = bio[0].find_all("div", class_="c-bio__text")[2].text.strip() if bio[0].find_all("div", class_="c-bio__text")[2] else "N/A"
            except:
                stats["Weight"] = "N/A"
            # Segunda linha de informações
            try:
                stats["UFC_Debut"] = bio[1].find_all("div", class_="c-bio__text")[0].text.strip() if bio[1].find_all("div", class_="c-bio__text")[0] else "N/A"
            except:
                stats["UFC_Debut"] = "N/A"
            try:
                stats["Reach"] = bio[1].find_all("div", class_="c-bio__text")[1].text.strip() if bio[1].find_all("div", class_="c-bio__text")[1] else "N/A"
            except:
                stats["Reach"] = "N/A"
            try:
                stats["Leg_Reach"] = bio[1].find_all("div", class_="c-bio__text")[2].text.strip() if bio[1].find_all("div", class_="c-bio__text")[2] else "N/A"
            except:
                stats["Leg_Reach"] = "N/A"       
                    
        defense_stats = soup.find_all("div", class_="c-stat-compare__group")
        if defense_stats:
            # Golpes Significativos
            try:
                stats["Significant_Strikes_Landed_Per_Min"] = defense_stats[0].find("div", class_="c-stat-compare__number").text.strip() if defense_stats[0].find("div", class_="c-stat-compare__number") else "N/A"
            except:
                stats["Significant_Strikes_Landed_Per_Min"] = "N/A"
            try:
                stats["Significant_Strikes_Absorbed_Per_Min"] = defense_stats[1].find("div", class_="c-stat-compare__number").text.strip() if defense_stats[1].find("div", class_="c-stat-compare__number") else "N/A"
            except:
                stats["Significant_Strikes_Absorbed_Per_Min"] = "N/A"

            # Quedas e Finalizações
            try:
                stats["Takedowns_Average_Per_15min"] = defense_stats[2].find("div", class_="c-stat-compare__number").text.strip() if defense_stats[2].find("div", class_="c-stat-compare__number") else "N/A"
            except:
                stats["Takedowns_Average_Per_15min"] = "N/A"
            try:
                stats["Submissions_Average_Per_15min"] = defense_stats[3].find("div", class_="c-stat-compare__number").text.strip() if defense_stats[3].find("div", class_="c-stat-compare__number") else "N/A"
            except:
                stats["Submissions_Average_Per_15min"] = "N/A"

            # Defesas
            try:
                stats["Significant_Strike_Defense_Percentage"] = defense_stats[4].find("div", class_="c-stat-compare__number").text.strip().replace(" ", "").replace("\n", "").replace("%", "") if defense_stats[4].find("div", class_="c-stat-compare__number") else "N/A"
            except:
                stats["Significant_Strike_Defense_Percentage"] = "N/A"
            try:
                stats["Takedown_Defense"] = defense_stats[5].find("div", class_="c-stat-compare__number").text.strip().replace(" ", "").replace("\n", "").replace("%", "") if defense_stats[5].find("div", class_="c-stat-compare__number") else "N/A"
            except:
                stats["Takedown_Defense"] = "N/A"

            # Knockdowns e Tempo
            try:
                stats["Knockdowns_Average"] = defense_stats[6].find("div", class_="c-stat-compare__number").text.strip() if defense_stats[6].find("div", class_="c-stat-compare__number") else "N/A"
            except:
                stats["Knockdowns_Average"] = "N/A"
            try:
                stats["Average_Fight_Time"] = defense_stats[7].find("div", class_="c-stat-compare__number").text.strip() if defense_stats[7].find("div", class_="c-stat-compare__number") else "N/A"
            except:
                stats["Average_Fight_Time"] = "N/A"

        significant_strikes_area = soup.find("div", class_="c-stat-body")
        if significant_strikes_area:

            head_percent = significant_strikes_area.find("text", id="e-stat-body_x5F__x5F_head_percent")
            head_value = significant_strikes_area.find("text", id="e-stat-body_x5F__x5F_head_value")
            stats["Head_Strikes_Percent"] = head_percent.text.strip().replace("%", "") if head_percent else "N/A"
            stats["Head_Strikes_Count"] = head_value.text.strip() if head_value else "N/A"

            # Golpes no corpo
            body_percent = significant_strikes_area.find("text", id="e-stat-body_x5F__x5F_body_percent")
            body_value = significant_strikes_area.find("text", id="e-stat-body_x5F__x5F_body_value")
            stats["Body_Strikes_Percent"] = body_percent.text.strip().replace("%", "") if body_percent else "N/A"
            stats["Body_Strikes_Count"] = body_value.text.strip() if body_value else "N/A"

            # Golpes nas pernas
            leg_percent = significant_strikes_area.find("text", id="e-stat-body_x5F__x5F_leg_percent")
            leg_value = significant_strikes_area.find("text", id="e-stat-body_x5F__x5F_leg_value")
            stats["Leg_Strikes_Percent"] = leg_percent.text.strip().replace("%", "") if leg_percent else "N/A"
            stats["Leg_Strikes_Count"] = leg_value.text.strip() if leg_value else "N/A"
        else:
                stats["Head_Strikes_Percent"] = stats["Head_Strikes_Count"] = "N/A"
                stats["Body_Strikes_Percent"] = stats["Body_Strikes_Count"] = "N/A"
                stats["Leg_Strikes_Percent"] = stats["Leg_Strikes_Count"] = "N/A"
        
        # Encontrar a seção de vitórias por método
        win_by_method_section = soup.find_all("div", class_="c-stat-3bar c-stat-3bar--no-chart")
        if win_by_method_section:
            for section in win_by_method_section:
                title = section.find("h2", class_="c-stat-3bar__title")
                if title and "Win by Method" in title.text:
                    method_groups = section.find_all("div", class_="c-stat-3bar__group")
                    for method in method_groups:
                        method_label = method.find("div", class_="c-stat-3bar__label").text.strip()
                        if method_label in ['KO/TKO', 'DEC', 'FIN']:
                            value_text = method.find("div", class_="c-stat-3bar__value").text.strip()
                            count, percentage = value_text.replace(" %", "%").split()
                            percentage = percentage.strip('()%')
                            stats[f"Win by {method_label} Count"] = count
                            stats[f"Win by {method_label} Percentage"] = percentage
                    break
        else:
            stats["Win by KO/TKO Count"] = stats["Win by KO/TKO Percentage"] \
                    = stats["Win by DEC Count"] = stats["Win by DEC Percentage"] \
                    = stats["Win by FIN Count"] = stats["Win by FIN Percentage"] = "N/A"

        fighting_style_divs = soup.find_all("div", class_="c-bio__field c-bio__field--border-bottom-small-screens")

        if fighting_style_divs:
            fighting_style = next(
                (tag.find("div", class_="c-bio__text").text.strip()
                for tag in fighting_style_divs
                if tag.find("div", class_="c-bio__label").text.strip() == "Estilo de luta"),
                "N/A"
            )
            stats['fighting_style'] = fighting_style
        else:
            stats['fighting_style'] = "N/A"

        location_divs = soup.find_all("div", class_="c-bio__field c-bio__field--border-bottom-small-screens")
        if location_divs:
            location = next(
                (tag.find("div", class_="c-bio__text").text.strip()
                for tag in location_divs
                if tag.find("div", class_="c-bio__label").text.strip() == "Cidade natal"),
                "N/A"
            )
            stats['hometown'] = location
        else:
            stats['hometown'] = "N/A"

        return stats
    except Exception as e:
        print(f"Erro ao processar perfil do atleta: {url}. Erro: {e}")
        return {}


# Função para percorrer as páginas e coletar todos os atletas
def scrape_all_athletes(base_url):
    all_athletes = []
    page = 1
    
    while True:
        print(f"Scraping page {page}...")
        url = f"{base_url}?page={page}"
        athletes = scrape_athletes_from_page(url)
        
        if not athletes:
            break
        
        all_athletes.extend(athletes)
        page += 1
        
        # Pausar entre requisições para evitar sobrecarga no servidor
        time.sleep(2)  # Atraso de 2 segundos entre requisições
        
    return all_athletes

# Função para salvar os dados em um arquivo CSV
def save_to_csv(data, filename="ufc_athletes.csv"):
    if data:
        keys = data[0].keys()
        with open(filename, "w", newline="", encoding="utf-8") as file:
            writer = csv.DictWriter(file, fieldnames=keys)
            writer.writeheader()
            writer.writerows(data)
    else:
        print("Nenhum dado para salvar.")

# Main
if __name__ == "__main__":
    print("Iniciando o scraping...")
    all_athletes = scrape_all_athletes(BASE_URL)
    print(f"Total de atletas extraídos: {len(all_athletes)}")
    save_to_csv(all_athletes)
    print("Dados salvos em 'ufc_athletes.csv'.")


Iniciando o scraping...
Scraping page 1...
Scraping page 2...
Scraping page 3...
Scraping page 4...
Scraping page 5...
Scraping page 6...
Scraping page 7...
Scraping page 8...
Scraping page 9...
Scraping page 10...
Scraping page 11...
Scraping page 12...
Scraping page 13...
Scraping page 14...
Scraping page 15...
Scraping page 16...
Scraping page 17...
Scraping page 18...
Scraping page 19...
Scraping page 20...
Scraping page 21...
Scraping page 22...
Scraping page 23...
Scraping page 24...
Scraping page 25...
Scraping page 26...
Scraping page 27...
Scraping page 28...
Scraping page 29...
Scraping page 30...
Scraping page 31...
Scraping page 32...
Scraping page 33...
Scraping page 34...
Scraping page 35...
Scraping page 36...
Scraping page 37...
Scraping page 38...
Scraping page 39...
Scraping page 40...
Scraping page 41...
Scraping page 42...
Scraping page 43...
Scraping page 44...
Scraping page 45...
Scraping page 46...
Scraping page 47...
Scraping page 48...
Scraping page 49...
Scrap

KeyboardInterrupt: 

In [ ]:
import requests
from bs4 import BeautifulSoup
import csv
import time

def scrape_athlete_profile(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")
    
    try:
        # Informações detalhadas
        stats = {}
        
        overlap_stats = soup.find_all("dl", class_="c-overlap__stats")
        for stat in overlap_stats:
            label = stat.find("dt").text.strip()
            value = stat.find("dd").text.strip()
            
            if label == "Golpes Sig. Conectados":
                stats["Significant Strikes Landed"] = value
            elif label == "Golpes Sig. Desferidos":
                stats["Significant Strikes Attempted"] = value
            elif label == "Quedas aplicadas":
                stats["Takedowns Landed"] = value
            elif label == "Tentativas de queda":
                stats["Takedowns Attempted"] = value

        bio = soup.find_all("div", class_="c-bio__row--3col")
        if bio:
            # Primeira linha de informações
            try:
                stats["Age"] = bio[0].find("div", class_="field__item").text.strip() if bio[0].find("div", class_="field__item") else "N/A"
            except:
                stats["Age"] = "N/A"
            try:
                stats["Height"] = bio[0].find_all("div", class_="c-bio__text")[1].text.strip() if bio[0].find_all("div", class_="c-bio__text")[1] else "N/A"
            except:
                stats["Height"] = "N/A"
            try:
                stats["Weight"] = bio[0].find_all("div", class_="c-bio__text")[2].text.strip() if bio[0].find_all("div", class_="c-bio__text")[2] else "N/A"
            except:
                stats["Weight"] = "N/A"
            # Segunda linha de informações
            try:
                stats["UFC_Debut"] = bio[1].find_all("div", class_="c-bio__text")[0].text.strip() if bio[1].find_all("div", class_="c-bio__text")[0] else "N/A"
            except:
                stats["UFC_Debut"] = "N/A"
            try:
                stats["Reach"] = bio[1].find_all("div", class_="c-bio__text")[1].text.strip() if bio[1].find_all("div", class_="c-bio__text")[1] else "N/A"
            except:
                stats["Reach"] = "N/A"
            try:
                stats["Leg_Reach"] = bio[1].find_all("div", class_="c-bio__text")[2].text.strip() if bio[1].find_all("div", class_="c-bio__text")[2] else "N/A"
            except:
                stats["Leg_Reach"] = "N/A"       
                    
        defense_stats = soup.find_all("div", class_="c-stat-compare__group")
        if defense_stats:
            # Golpes Significativos
            try:
                stats["Significant_Strikes_Landed_Per_Min"] = defense_stats[0].find("div", class_="c-stat-compare__number").text.strip() if defense_stats[0].find("div", class_="c-stat-compare__number") else "N/A"
            except:
                stats["Significant_Strikes_Landed_Per_Min"] = "N/A"
            try:
                stats["Significant_Strikes_Absorbed_Per_Min"] = defense_stats[1].find("div", class_="c-stat-compare__number").text.strip() if defense_stats[1].find("div", class_="c-stat-compare__number") else "N/A"
            except:
                stats["Significant_Strikes_Absorbed_Per_Min"] = "N/A"

            # Quedas e Finalizações
            try:
                stats["Takedowns_Average_Per_15min"] = defense_stats[2].find("div", class_="c-stat-compare__number").text.strip() if defense_stats[2].find("div", class_="c-stat-compare__number") else "N/A"
            except:
                stats["Takedowns_Average_Per_15min"] = "N/A"
            try:
                stats["Submissions_Average_Per_15min"] = defense_stats[3].find("div", class_="c-stat-compare__number").text.strip() if defense_stats[3].find("div", class_="c-stat-compare__number") else "N/A"
            except:
                stats["Submissions_Average_Per_15min"] = "N/A"

            # Defesas
            try:
                stats["Significant_Strike_Defense_Percentage"] = defense_stats[4].find("div", class_="c-stat-compare__number").text.strip().replace(" ", "").replace("\n", "").replace("%", "") if defense_stats[4].find("div", class_="c-stat-compare__number") else "N/A"
            except:
                stats["Significant_Strike_Defense_Percentage"] = "N/A"
            try:
                stats["Takedown_Defense"] = defense_stats[5].find("div", class_="c-stat-compare__number").text.strip().replace(" ", "").replace("\n", "").replace("%", "") if defense_stats[5].find("div", class_="c-stat-compare__number") else "N/A"
            except:
                stats["Takedown_Defense"] = "N/A"

            # Knockdowns e Tempo
            try:
                stats["Knockdowns_Average"] = defense_stats[6].find("div", class_="c-stat-compare__number").text.strip() if defense_stats[6].find("div", class_="c-stat-compare__number") else "N/A"
            except:
                stats["Knockdowns_Average"] = "N/A"
            try:
                stats["Average_Fight_Time"] = defense_stats[7].find("div", class_="c-stat-compare__number").text.strip() if defense_stats[7].find("div", class_="c-stat-compare__number") else "N/A"
            except:
                stats["Average_Fight_Time"] = "N/A"

        significant_strikes_area = soup.find("div", class_="c-stat-body")
        if significant_strikes_area:

            head_percent = significant_strikes_area.find("text", id="e-stat-body_x5F__x5F_head_percent")
            head_value = significant_strikes_area.find("text", id="e-stat-body_x5F__x5F_head_value")
            stats["Head_Strikes_Percent"] = head_percent.text.strip().replace("%", "") if head_percent else "N/A"
            stats["Head_Strikes_Count"] = head_value.text.strip() if head_value else "N/A"

            # Golpes no corpo
            body_percent = significant_strikes_area.find("text", id="e-stat-body_x5F__x5F_body_percent")
            body_value = significant_strikes_area.find("text", id="e-stat-body_x5F__x5F_body_value")
            stats["Body_Strikes_Percent"] = body_percent.text.strip().replace("%", "") if body_percent else "N/A"
            stats["Body_Strikes_Count"] = body_value.text.strip() if body_value else "N/A"

            # Golpes nas pernas
            leg_percent = significant_strikes_area.find("text", id="e-stat-body_x5F__x5F_leg_percent")
            leg_value = significant_strikes_area.find("text", id="e-stat-body_x5F__x5F_leg_value")
            stats["Leg_Strikes_Percent"] = leg_percent.text.strip().replace("%", "") if leg_percent else "N/A"
            stats["Leg_Strikes_Count"] = leg_value.text.strip() if leg_value else "N/A"
        else:
                stats["Head_Strikes_Percent"] = stats["Head_Strikes_Count"] = "N/A"
                stats["Body_Strikes_Percent"] = stats["Body_Strikes_Count"] = "N/A"
                stats["Leg_Strikes_Percent"] = stats["Leg_Strikes_Count"] = "N/A"
        
        # Encontrar a seção de vitórias por método
        win_by_method_section = soup.find_all("div", class_="c-stat-3bar c-stat-3bar--no-chart")
        if win_by_method_section:
            for section in win_by_method_section:
                title = section.find("h2", class_="c-stat-3bar__title")
                if title and "Win by Method" in title.text:
                    method_groups = section.find_all("div", class_="c-stat-3bar__group")
                    for method in method_groups:
                        method_label = method.find("div", class_="c-stat-3bar__label").text.strip()
                        if method_label in ['KO/TKO', 'DEC', 'FIN']:
                            value_text = method.find("div", class_="c-stat-3bar__value").text.strip()
                            count, percentage = value_text.replace(" %", "%").split()
                            percentage = percentage.strip('()%')
                            stats[f"Win by {method_label} Count"] = count
                            stats[f"Win by {method_label} Percentage"] = percentage
                    break
        else:
            stats["Win by KO/TKO Count"] = stats["Win by KO/TKO Percentage"] \
                    = stats["Win by DEC Count"] = stats["Win by DEC Percentage"] \
                    = stats["Win by FIN Count"] = stats["Win by FIN Percentage"] = "N/A"

        fighting_style_divs = soup.find_all("div", class_="c-bio__field c-bio__field--border-bottom-small-screens")

        if fighting_style_divs:
            fighting_style = next(
                (tag.find("div", class_="c-bio__text").text.strip()
                for tag in fighting_style_divs
                if tag.find("div", class_="c-bio__label").text.strip() == "Estilo de luta"),
                "N/A"
            )
            stats['fighting_style'] = fighting_style
        else:
            stats['fighting_style'] = "N/A"

        location_divs = soup.find_all("div", class_="c-bio__field c-bio__field--border-bottom-small-screens")
        if location_divs:
            location = next(
                (tag.find("div", class_="c-bio__text").text.strip()
                for tag in location_divs
                if tag.find("div", class_="c-bio__label").text.strip() == "Cidade natal"),
                "N/A"
            )
            stats['hometown'] = location
        else:
            stats['hometown'] = "N/A"

        return stats
    except Exception as e:
        print(f"Erro ao processar perfil do atleta: {url}. Erro: {e}")
        return {}
    
scrape_athlete_profile("https://www.ufc.com.br/athlete/ilia-topuria")

In [108]:
overlap_stats = soup.find_all("dl", class_="c-overlap__stats")
for stat in overlap_stats:
    label = stat.find("dt").text.strip()
    value = stat.find("dd").text.strip()
    
    if label == "Golpes Sig. Conectados":
        stats["Significant Strikes Landed"] = value
    elif label == "Golpes Sig. Desferidos":
        stats["Significant Strikes Attempted"] = value
    elif label == "Quedas aplicadas":
        stats["Takedowns Landed"] = value
    elif label == "Tentativas de queda":
        stats["Takedowns Attempted"] = value


In [109]:
bio = soup.find_all("div", class_="c-bio__row--3col")
if bio:
    # Primeira linha de informações
    stats["Age"] = bio[0].find("div", class_="field__item").text.strip() if bio[0].find("div", class_="field__item") else "N/A"
    stats["Height"] = bio[0].find_all("div", class_="c-bio__text")[1].text.strip()
    stats["Weight"] = bio[0].find_all("div", class_="c-bio__text")[2].text.strip()
    
    # Segunda linha de informações
    stats["UFC_Debut"] = bio[1].find_all("div", class_="c-bio__text")[0].text.strip()
    stats["Reach"] = bio[1].find_all("div", class_="c-bio__text")[1].text.strip()
    stats["Leg_Reach"] = bio[1].find_all("div", class_="c-bio__text")[2].text.strip()


In [110]:
defense_stats = soup.find_all("div", class_="c-stat-compare__group")
# Golpes Significativos
stats["Significant_Strikes_Landed_Per_Min"] = defense_stats[0].find("div", class_="c-stat-compare__number").text.strip()
stats["Significant_Strikes_Absorbed_Per_Min"] = defense_stats[1].find("div", class_="c-stat-compare__number").text.strip()

# Quedas e Finalizações
stats["Takedowns_Average_Per_15min"] = defense_stats[2].find("div", class_="c-stat-compare__number").text.strip()
stats["Submissions_Average_Per_15min"] = defense_stats[3].find("div", class_="c-stat-compare__number").text.strip()

# Defesas
stats["Significant_Strike_Defense"] = defense_stats[4].find("div", class_="c-stat-compare__number").text.strip().replace(" ", "").replace("\n", "")
stats["Takedown_Defense"] = defense_stats[5].find("div", class_="c-stat-compare__number").text.strip().replace(" ", "").replace("\n", "")

# Knockdowns e Tempo
stats["Knockdowns_Average"] = defense_stats[6].find("div", class_="c-stat-compare__number").text.strip()
stats["Average_Fight_Time"] = defense_stats[7].find("div", class_="c-stat-compare__number").text.strip()


In [111]:
significant_strikes_area = soup.find("div", class_="c-stat-body")
if significant_strikes_area:

    head_percent = significant_strikes_area.find("text", id="e-stat-body_x5F__x5F_head_percent")
    head_value = significant_strikes_area.find("text", id="e-stat-body_x5F__x5F_head_value")
    stats["Head_Strikes_Percent"] = head_percent.text.strip() if head_percent else "N/A"
    stats["Head_Strikes_Count"] = head_value.text.strip() if head_value else "N/A"

    # Golpes no corpo
    body_percent = significant_strikes_area.find("text", id="e-stat-body_x5F__x5F_body_percent")
    body_value = significant_strikes_area.find("text", id="e-stat-body_x5F__x5F_body_value")
    stats["Body_Strikes_Percent"] = body_percent.text.strip() if body_percent else "N/A"
    stats["Body_Strikes_Count"] = body_value.text.strip() if body_value else "N/A"

    # Golpes nas pernas
    leg_percent = significant_strikes_area.find("text", id="e-stat-body_x5F__x5F_leg_percent")
    leg_value = significant_strikes_area.find("text", id="e-stat-body_x5F__x5F_leg_value")
    stats["Leg_Strikes_Percent"] = leg_percent.text.strip() if leg_percent else "N/A"
    stats["Leg_Strikes_Count"] = leg_value.text.strip() if leg_value else "N/A"
else:
        stats["Head_Strikes_Percent"] = stats["Head_Strikes_Count"] = "N/A"
        stats["Body_Strikes_Percent"] = stats["Body_Strikes_Count"] = "N/A"
        stats["Leg_Strikes_Percent"] = stats["Leg_Strikes_Count"] = "N/A"
        

In [112]:
# Encontrar a seção de vitórias por método
win_by_method_section = soup.find_all("div", class_="c-stat-3bar c-stat-3bar--no-chart")
for section in win_by_method_section:
    title = section.find("h2", class_="c-stat-3bar__title")
    if title and "Win by Method" in title.text:
        method_groups = section.find_all("div", class_="c-stat-3bar__group")
        for method in method_groups:
            method_label = method.find("div", class_="c-stat-3bar__label").text.strip()
            if method_label in ['KO/TKO', 'DEC', 'FIN']:
                value_text = method.find("div", class_="c-stat-3bar__value").text.strip()
                count, percentage = value_text.split()
                percentage = percentage.strip('()%')
                stats[f"Win by {method_label} Count"] = count
                stats[f"Win by {method_label} Percentage"] = percentage
        break

In [126]:
fighting_style_divs = soup.find_all("div", class_="c-bio__field c-bio__field--border-bottom-small-screens")
fighting_style = next(
    (tag.find("div", class_="c-bio__text").text.strip()
     for tag in fighting_style_divs
     if tag.find("div", class_="c-bio__label").text.strip() == "Estilo de luta"),
    "N/A"
)
stats['fighting_style'] = fighting_style

In [130]:
location_divs = soup.find_all("div", class_="c-bio__field c-bio__field--border-bottom-small-screens")
location = next(
    (tag.find("div", class_="c-bio__text").text.strip()
     for tag in location_divs
     if tag.find("div", class_="c-bio__label").text.strip() == "Cidade natal"),
    "N/A"
)
stats['hometown'] = location

In [ ]:
stats

In [1]:
import pandas as pd
df = pd.read_csv('/home/gassuncao/mma-status/mma-stats/data/raw/ufc_athletes.csv')
df.columns


Index(['Athlete ID', 'Name', 'Nickname', 'Wins', 'Losses', 'Draws',
       'Weight Class', 'Significant Strikes Landed',
       'Significant Strikes Attempted', 'Takedowns Landed',
       'Takedowns Attempted', 'Age', 'Height', 'Weight', 'UFC_Debut', 'Reach',
       'Leg_Reach', 'Significant_Strikes_Landed_Per_Min',
       'Significant_Strikes_Absorbed_Per_Min', 'Takedowns_Average_Per_15min',
       'Submissions_Average_Per_15min',
       'Significant_Strike_Defense_Percentage', 'Takedown_Defense',
       'Knockdowns_Average', 'Average_Fight_Time', 'Head_Strikes_Percent',
       'Head_Strikes_Count', 'Body_Strikes_Percent', 'Body_Strikes_Count',
       'Leg_Strikes_Percent', 'Leg_Strikes_Count', 'Win by KO/TKO Count',
       'Win by KO/TKO Percentage', 'Win by DEC Count', 'Win by DEC Percentage',
       'Win by FIN Count', 'Win by FIN Percentage', 'fighting_style',
       'hometown'],
      dtype='object')

In [10]:
df[df["Athlete ID"] == "islam-makhachev"]

,Athlete ID,Name,Nickname,Wins,Losses,Draws,Weight Class,Significant Strikes Landed,Significant Strikes Attempted,Takedowns Landed,...,Leg_Strikes_Percent,Leg_Strikes_Count,Win by KO/TKO Count,Win by KO/TKO Percentage,Win by DEC Count,Win by DEC Percentage,Win by FIN Count,Win by FIN Percentage,fighting_style,hometown
1617,islam-makhachev,Islam Makhachev,NaN,27,1,0,Peso-leve,455.0,773.0,13.0,...,4.0,17.0,5.0,19.0,9.0,33.0,13.0,48.0,Sambo,"Dagestan Republic, Rússia"


In [4]:
import requests
from bs4 import BeautifulSoup
import csv
import time


response = requests.get("https://www.ufc.com.br/athlete/israel-adesanya#athlete-record")
soup = BeautifulSoup(response.text, "html.parser")

In [5]:
print(soup)


<!DOCTYPE html>

<html dir="ltr" lang="pt-br" prefix="og: https://ogp.me/ns#">
<head>
<meta charset="utf-8"/>
<meta content='Israel "The Last Stylebender" is a New Zealander professional mixed martial artist in the UFC middleweight division, and the former UFC middleweight champion. Get the latest UFC breaking news, fight night results, MMA records and stats, highlights, photos, videos and more.' name="description"/>
<link href="https://www.ufc.com.br/athlete/israel-adesanya" rel="canonical"/>
<meta content="origin" name="referrer"/>
<meta content="Israel Adesanya | UFC" property="og:title"/>
<meta content='Israel "The Last Stylebender" is a New Zealander professional mixed martial artist in the UFC middleweight division, and the former UFC middleweight champion. Get the latest UFC breaking news, fight night results, MMA records and stats, highlights, photos, videos and more.' property="og:description"/>
<meta content="https://dmxg5wxfqgb4u.cloudfront.net/2025-01/5/ADESANYA_ISRAEL_08-

In [7]:
##*************##
# Melhor versao #
##*************##
import requests
from bs4 import BeautifulSoup
import time

def get_fight_history(athlete_id):
    base_url = f"https://www.ufc.com.br/athlete/{athlete_id}"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    
    fight_history = []
    page_url = base_url + "#athlete-record"
    
    while page_url:
        print(f"Coletando dados da página: {page_url}")
        
        # Faz a requisição
        response = requests.get(page_url, headers=headers)
        if response.status_code != 200:
            print(f"Erro ao acessar a página: {page_url}")
            break

        # Faz o parsing do HTML
        soup = BeautifulSoup(response.text, 'html.parser')

        # Localiza o container das lutas
        # fight_elements = soup.find_all('div', class_='c-card-event--athlete-results__matchup')
        fight_elements = soup.find_all('article', class_='c-card-event--athlete-results')
        # fight_elements = soup.find_all('li', class_='l-listing__item')
        
        # Itera sobre cada luta encontrada
        for fight in fight_elements:
            try:
                opponent_id = next(id for id in [item["href"].split('/')[-1] for item in fight.find('div', class_='c-card-event--athlete-results__info').find_all('a')] if id != athlete_id)               
                round_info = fight.find('div', string='Round')
                if round_info:
                    round_info = round_info.find_next('div').text.strip()
                else:
                    round_info = "N/A"
    
                time_info = fight.find('div', string='Tempo')
                if time_info:
                    time_info = time_info.find_next('div').text.strip()
                else:
                    time_info = "N/A"

                method_info = fight.find('div', string='Método')
                if method_info:
                    method_info = method_info.find_next('div').text.strip()
                else:
                    method_info = "N/A"

                link_element = fight.find('div', class_='c-card-event--athlete-results__image c-card-event--athlete-results__red-image win')
              
                if not link_element:
                    link_element = fight.find('div', class_='c-card-event--athlete-results__image c-card-event--athlete-results__blue-image win')
         
                if not link_element:
                    victory_athlete_id = "N/A"
                else:
                    href = link_element.find('a', href=True)['href']
                    victory_athlete_id = href.split('/')[-1]
                    if victory_athlete_id == athlete_id:
                        result = "Victory"
                    if victory_athlete_id != athlete_id:
                        result = "Loss"
                    if victory_athlete_id == "N/A":
                        result = "N/A"
                fight_history.append({
                    "opponent_id": opponent_id,
                    "round": round_info,
                    "time": time_info,
                    "method": method_info,
                    "result": result
                })
            except Exception as e:
                print(f"Erro ao processar luta: {e}")

        # Verifica se há mais páginas
        next_page = soup.find('a', class_='button', title='Load more items')
        if next_page:
            page_url = base_url + next_page['href']
            time.sleep(2)  # Delay para evitar sobrecarga no servidor
        else:
            page_url = None

    return fight_history

# Lista de IDs dos atletas
athlete_ids = ["michael-bisping"]

# Coleta as informações para cada atleta
all_fights = {}
for athlete_id in athlete_ids:
    print(f"Coletando histórico de lutas para: {athlete_id}")
    all_fights[athlete_id] = get_fight_history(athlete_id)

# Exibe o resultado
for athlete, fights in all_fights.items():
    print(f"\nHistórico de lutas para {athlete}:")
    for fight in fights:
        print(fight)


Coletando histórico de lutas para: michael-bisping
Coletando dados da página: https://www.ufc.com.br/athlete/michael-bisping#athlete-record


Coletando dados da página: https://www.ufc.com.br/athlete/michael-bisping?page=1
Coletando dados da página: https://www.ufc.com.br/athlete/michael-bisping?page=2
Coletando dados da página: https://www.ufc.com.br/athlete/michael-bisping?page=3
Coletando dados da página: https://www.ufc.com.br/athlete/michael-bisping?page=4
Coletando dados da página: https://www.ufc.com.br/athlete/michael-bisping?page=5
Coletando dados da página: https://www.ufc.com.br/athlete/michael-bisping?page=6
Coletando dados da página: https://www.ufc.com.br/athlete/michael-bisping?page=7
Coletando dados da página: https://www.ufc.com.br/athlete/michael-bisping?page=8
Coletando dados da página: https://www.ufc.com.br/athlete/michael-bisping?page=9

Histórico de lutas para michael-bisping:
{'opponent_id': 'kelvin-gastelum', 'round': '1', 'time': '02:29', 'method': 'KO/TKO', 'result': 'Loss'}
{'opponent_id': 'georges-st-pierre', 'round': '3', 'time': '4:23', 'method': 'Submissão', 'result': 'Loss'}
{'opponent_id': '

In [29]:
import requests
from bs4 import BeautifulSoup
import time

def get_fight_history(athlete_id):
    base_url = f"https://www.ufc.com.br/athlete/{athlete_id}"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    
    fight_history = []
    page_url = base_url + "#athlete-record"
    
    while page_url:
        print(f"Coletando dados da página: {page_url}")
        
        # Faz a requisição
        response = requests.get(page_url, headers=headers)
        if response.status_code != 200:
            print(f"Erro ao acessar a página: {page_url}")
            break

        # Faz o parsing do HTML
        soup = BeautifulSoup(response.text, 'html.parser')

        # Localiza o container das lutas
        # fight_elements = soup.find_all('div', class_='c-card-event--athlete-results__matchup')
        fight_elements = soup.find_all('article', class_='c-card-event--athlete-results')
        # fight_elements = soup.find_all('li', class_='l-listing__item')
        
        # Itera sobre cada luta encontrada
        for fight in fight_elements:
            try:
                opponent_id = next(id for id in [item["href"].split('/')[-1] for item in fight.find('div', class_='c-card-event--athlete-results__info').find_all('a')] if id != athlete_id)               
                round_info = fight.find('div', string='Round').find_next('div').text.strip()
                time_info = fight.find('div', string='Tempo').find_next('div').text.strip()
                method_info = fight.find('div', string='Método').find_next('div').text.strip()
                link_element = fight.find('div', class_='c-card-event--athlete-results__image c-card-event--athlete-results__red-image win')
                
                if not link_element:
                    link_element = fight.find('div', class_='c-card-event--athlete-results__image c-card-event--athlete-results__blue-image win')
         
                if not link_element:
                    victory_athlete_id = "N/A"
                else:
                    href = link_element.find('a', href=True)['href']
                    victory_athlete_id = href.split('/')[-1]

                fight_history.append({
                    "opponent_id": opponent_id,
                    "round": round_info,
                    "time": time_info,
                    "method": method_info,
                    "result": "Victory" if victory_athlete_id == athlete_id else "Loss"
                })
            except Exception as e:
                print(f"Erro ao processar luta: {e}")

        # Verifica se há mais páginas
        next_page = soup.find('a', class_='button', title='Load more items')
        if next_page:
            page_url = base_url + next_page['href']
            time.sleep(2)  # Delay para evitar sobrecarga no servidor
        else:
            page_url = None

    return fight_history

# Lista de IDs dos atletas
athlete_ids = ["israel-adesanya"]

# Coleta as informações para cada atleta
all_fights = {}
for athlete_id in athlete_ids:
    print(f"Coletando histórico de lutas para: {athlete_id}")
    all_fights[athlete_id] = get_fight_history(athlete_id)

# Exibe o resultado
for athlete, fights in all_fights.items():
    print(f"\nHistórico de lutas para {athlete}:")
    for fight in fights:
        print(fight)


Coletando histórico de lutas para: israel-adesanya
Coletando dados da página: https://www.ufc.com.br/athlete/israel-adesanya#athlete-record
Coletando dados da página: https://www.ufc.com.br/athlete/israel-adesanya?page=1
Coletando dados da página: https://www.ufc.com.br/athlete/israel-adesanya?page=2
Coletando dados da página: https://www.ufc.com.br/athlete/israel-adesanya?page=3
Coletando dados da página: https://www.ufc.com.br/athlete/israel-adesanya?page=4
Coletando dados da página: https://www.ufc.com.br/athlete/israel-adesanya?page=5

Histórico de lutas para israel-adesanya:
{'opponent_id': 'dricus-du-plessis', 'round': '4', 'time': '3:38', 'method': 'Submissão', 'result': 'Loss'}
{'opponent_id': 'sean-strickland', 'round': '5', 'time': '5:00', 'method': 'Decision - Unanimous', 'result': 'Loss'}
{'opponent_id': 'alex-pereira', 'round': '2', 'time': '4:21', 'method': 'KO/TKO', 'result': 'Victory'}
{'opponent_id': 'alex-pereira', 'round': '5', 'time': '2:01', 'method': 'KO/TKO', 're

In [3]:
import pandas as pd
df = pd.read_csv('/home/gassuncao/mma-status/mma-stats/data/raw/ufc_athletes.csv')
df.columns

df[df["Athlete ID"] == "sean-omalley"]

,Athlete ID,Name,Nickname,Wins,Losses,Draws,Weight Class,Significant Strikes Landed,Significant Strikes Attempted,Takedowns Landed,...,Leg_Strikes_Percent,Leg_Strikes_Count,Win by KO/TKO Count,Win by KO/TKO Percentage,Win by DEC Count,Win by DEC Percentage,Win by FIN Count,Win by FIN Percentage,fighting_style,hometown
1956,sean-omalley,Sean O'Malley,"""Suga""",18,2,0,Peso-galo,1038.0,1690.0,3.0,...,9.0,90.0,12.0,67.0,5.0,28.0,1.0,6.0,NaN,"Helena, Estados Unidos"


In [23]:
import requests
from bs4 import BeautifulSoup
import time

base_url = f"https://www.ufc.com.br/event/ufc-243#7998"
headers = {
"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

fight_history = []
page_url = base_url + "#athlete-record"



# Faz a requisição
response = requests.get(page_url, headers=headers)


# Faz o parsing do HTML
soup = BeautifulSoup(response.text, 'html.parser')

In [24]:
soup


<!DOCTYPE html>

<html dir="ltr" lang="pt-br" prefix="og: https://ogp.me/ns#">
<head>
<meta charset="utf-8"/>
<meta content="UFC is coming to Melbourne on October 5 at Marvel Stadium. Whittaker vs Adesanya main card. Discover how and where to watch." name="description"/>
<link href="https://www.ufc.com.br/event/ufc-243" rel="canonical"/>
<meta content="origin" name="referrer"/>
<meta content="UFC 243 | UFC" property="og:title"/>
<meta content="UFC is coming to Melbourne on October 5 at Marvel Stadium. Whittaker vs Adesanya main card. Discover how and where to watch." property="og:description"/>
<meta content="2019-07-19T15:16:22-0300" property="article:published_time"/>
<meta content="2020-03-12T09:18:24-0300" property="article:modified_time"/>
<meta content="UFC is coming to Melbourne on October 5 at Marvel Stadium. Whittaker vs Adesanya main card. Discover how and where to watch." name="twitter:description"/>
<meta content="UFC 243 | UFC" name="twitter:title"/>
<meta content="Drupal